# Modulos

In [ ]:
import pandas as pd # Manipulación de datos
import numpy as np # Operaciones numéricas
import win32com.client as win32 # Automatización de Office (Word/Outlook)
import os # Manejo de rutas y archivos
import gc # Recolección de basura para liberar memoria
from typing import Optional, Any # Para anotaciones de tipos en funciones
from pathlib import Path # Manejo de rutas de archivos de forma más robusta
from typing import Dict # Para anotaciones de tipos en funciones
import random # Para generar datos de prueba aleatorios
import time # Para medir tiempos de ejecución y simular retrasos
import pdfkit # Para generación de PDFs a partir de HTML (requiere wkhtmltopdf instalado)
from datetime import datetime # Para manejo de fechas y horas

# Rutas y variables

Este bloque soluciona la variabilidad de los entornos de trabajo (PC Oficina vs. PC Personal) y la evolución mensual de los contratos.

#### Decisiones de Ingeniería y Optimización:
1.  **Uso de `Pathlib`**: Se prefiere sobre `os.path` porque gestiona automáticamente las barras inclinadas (`/` vs `\`) y permite concatenar rutas de forma legible.
2.  **Detección de `Path.home()`**: Resuelve el problema de la ruta absoluta `C:\Users\crist\`. Al usar el "Home" del sistema, el script funcionará sin cambios sin importar el nombre del usuario de Windows.
3.  **Parametrización Estructurada**: Se separan las variables `CONTRATO`, `INFORME_NUM` y `MES_TEXTO`. Esto permite que el próximo mes solo cambies dos strings en la parte superior y toda la estructura de carpetas (incluyendo la carpeta de salida de PDFs) se actualice automáticamente.
4.  **Integridad de Directorios**: Se implementa `mkdir(parents=True)`, lo que garantiza que si la carpeta "Negados" no existe dentro de la ruta del informe actual, el script la cree antes de intentar guardar los PDFs, evitando errores de ejecución.

#### Variables Temporales Eliminadas:
* `USER_HOME`: Variable de sistema temporal.
* `gc.collect()`: Invocado para asegurar limpieza tras operaciones de sistema de archivos.

#### Advertencias:
* **Filtros por Municipio**: La lógica para generar los archivos PDF (ej. `85440_VILLANUEVA.pdf`) se alimentará de `FOLDER_NEGADOS` en proximos bloques.

In [ ]:
# ==========================================
# CONFIGURACIÓN DINÁMICA DE RUTAS (REVISADO)
# ==========================================

# 1. CONFIGURACIÓN DE IDENTIDAD Y ENRUTAMIENTO
CUENTA_ORIGEN: str = "aseg13@capresoc.onmicrosoft.com"
NOMBRE_MOSTRAR: str = "ASEGURAMIENTO REGIMEN SUBSIDIADO"

# 2. CONFIGURACIÓN DE POLÍTICAS DE ENVÍO (SEGURIDAD)
# Cambia a True para enviar copia oculta (BCC) a las EPS externas
ENVIAR_COPIA_EPS_EXTERNA: bool = True 
# Correo constante para auditoría interna de Ventanilla Única
CORREO_VENTANILLA_INTERNA: str = "capresocaeps@capresoca-casanare.gov.co"

# 3. Identificación del Home del Usuario (Resuelve el cambio entre PC de oficina y personal)
USER_HOME: Path = Path.home()

# 4. Base de OneDrive (Capresoca) - Se detecta dinámicamente
ONEDRIVE_PATH: Path = USER_HOME / "OneDrive - 891856000_CAPRESOCA E P S"

# 5. Parámetros del Proyecto (Variables de Control Mensual)
ANO_PROCESO: str = "2026"
CONTRATO: str = "CTO 102.2026"
INFORME_NUM: str = "05" # 🚩
MES_TEXTO: str = "Abril" # 🚩

# 6. Configuración de Binarios Externos (Centralizado para evitar ruido en lógica)
# Esta es la ruta al motor de renderizado PDF profesional
PATH_WKHTML_EXE: Path = Path(r"C:\Program Files\wkhtmltopdf\bin\wkhtmltopdf.exe")

# 7. Construcción de Rutas Base usando Pathlib
RUTA_PROYECTO_BASE: Path = (
    ONEDRIVE_PATH / 
    "Escritorio" / 
    "Yesid Rincón Z" / 
    "informes" / 
    ANO_PROCESO / 
    CONTRATO / 
    f"{CONTRATO.replace(' ', '')} Informe  #{INFORME_NUM}" / 
    "12 Actividad" / 
    "Bases de datos notificaciones telefonicas" / 
    "TRASLADOS A OTRAS EPS (SAT) - BDUA"
)

# 8. Rutas Específicas de Archivos e Imágenes
FILE_EXCEL_PATH: Path = RUTA_PROYECTO_BASE / f"Notificaciones_Traslados_Salida_04_2026.xlsx" # 🚩
FOLDER_NEGADOS: Path = RUTA_PROYECTO_BASE / "Correos"

# Imágenes estáticas
RUTA_IMG_CAPRE: Path = ONEDRIVE_PATH / "Imágenes" / "capre.png"
RUTA_IMG_SIPER: Path = ONEDRIVE_PATH / "Imágenes" / "Siper salud.png"

# 9. Creación de directorios de salida e Integridad de Rutas
if not FOLDER_NEGADOS.exists():
    FOLDER_NEGADOS.mkdir(parents=True, exist_ok=True)
    print(f"Directorio creado: {FOLDER_NEGADOS}")

# Validaciones Críticas (Calidad del Dato y del Entorno)
assert PATH_WKHTML_EXE.exists(), f"ERROR: Motor PDF no encontrado en {PATH_WKHTML_EXE}. Verifique la instalación."
assert FILE_EXCEL_PATH.exists(), f"ERROR: Excel no encontrado en {FILE_EXCEL_PATH}"
assert RUTA_IMG_CAPRE.exists(), f"ERROR: Imagen Capresoca no encontrada en {RUTA_IMG_CAPRE}"

print(f"Rutas configuradas correctamente. Motor PDF detectado en: {PATH_WKHTML_EXE.parent}")

# Limpieza de variables de configuración global para optimizar RAM
del USER_HOME
gc.collect()

# Carga de dataframes

### 1. Explicación Técnica del Bloque
Este bloque inicializa la ingesta de datos del archivo Excel `Notificaciones_Traslados_Salida_01_2026.xlsx` hacia un DataFrame principal denominado `df_salidas`. Implementa un enfoque defensivo, asegurando que los metadatos de las columnas estén limpios y listos para ser iterados o referenciados en la plantilla HTML sin riesgo de errores de tipografía.

### 2. Justificación de Decisiones de Optimización e Ingeniería
* **Preservación Estricta (`dtype=str`)**: Se evita delegar la inferencia de tipos a Pandas. Al forzar todo a `str`, evitamos que números de identificación largos se conviertan en notación científica o que códigos de municipios/EPS pierdan ceros a la izquierda.
* **Sanitización Vectorizada**: Se utiliza *List Comprehension* `[str(col).strip() for col in df_salidas.columns]` para limpiar los encabezados. Es el método computacionalmente más rápido y seguro en Python para transformar listas de strings.
* **Honestidad Brutal (Regla 5)**: Al no haber recibido explícitamente el nombre de la hoja de cálculo ni las columnas exactas que componen este nuevo archivo, se configuró `sheet_name=0` para capturar la primera hoja por defecto y no se construyeron variables derivadas (como el `MUNICIPIO_GRUPO` del script anterior) para evitar inyectar código basado en suposiciones.

### 3. Validaciones de Integridad Aplicadas
* **Control de Dimensionalidad**: Extracción y reporte automático del conteo de filas y columnas para certificar visualmente que la ingesta corresponde al volumen esperado.
* **Manejo de Excepciones**: Bloque `try-except` que previene el fallo silencioso y detiene la ejecución si el motor no logra montar el archivo en memoria.

### 4. Variables Temporales Eliminadas
* `total_filas` (int)
* `total_columnas` (int)
* `lista_columnas` (list)
* El recolector de basura (`gc.collect()`) fue invocado al finalizar para reclamar la RAM de estos punteros.

### 5. Advertencias
* **Nombre de Hoja**: Si tu archivo Excel contiene múltiples hojas y la data de interés no está en la primera posición (índice 0), la carga capturará datos incorrectos.
* **Falta de Llaves de Enrutamiento**: El script anterior agrupaba por `MUNICIPIO_GRUPO` usando `MNC_ID` y `Nombre_Municipio`. Dado que este es un proceso nuevo (Salidas S4-R4), no se ha creado esta llave. 

### 6. Validaciones Sugeridas
1. Revisa el output de la consola ("Estructura detectada") y confirma que las columnas necesarias para el asunto del correo, destinatario (`correo_electronico` o equivalente) y campos de la plantilla Word existan exactamente con esos nombres.
2. Confírmame en tu próxima instrucción qué columnas usaremos para identificar los correos válidos y si debemos crear una agrupación por municipio o EPS destino antes de armar el bloque de limpieza de datos.

In [ ]:
# ==========================================
# CARGA INTEGRAL Y SEGURA DEL DATAFRAME
# ==========================================

try:
    # 1. Carga masiva de la primera hoja del Excel (Traslados de Salida).
    # JUSTIFICACIÓN: Se utiliza dtype=str innegociablemente para preservar la integridad de todos 
    # los códigos (ej. ceros a la izquierda en IDs, códigos de EPS, teléfonos).
    # Se usa sheet_name=0 asumiendo que los datos están en la primera hoja al no tener 
    # un nombre de hoja explícitamente definido en las instrucciones.
    df_salidas: pd.DataFrame = pd.read_excel(
        FILE_EXCEL_PATH, 
        sheet_name= 'Consolidado_Salidas', 
        dtype=str,
        engine='openpyxl'
    )
    
    # 2. Sanitización estandarizada de Headers
    # JUSTIFICACIÓN: Previene errores de KeyError en futuros cruces o plantillas HTML 
    # causados por espacios invisibles accidentales (ej. "correo " vs "correo").
    df_salidas.columns = [str(col).strip() for col in df_salidas.columns]
    
    # 3. Auditoría inicial de integridad y dimensionalidad
    total_filas: int = len(df_salidas)
    total_columnas: int = len(df_salidas.columns)
    lista_columnas: list = df_salidas.columns.tolist()

    print("--- REPORTE DE CARGA: TRASLADOS DE SALIDA ---")
    print(f"Total registros (filas) cargados: {total_filas}")
    print(f"Total variables (columnas) detectadas: {total_columnas}")
    print(f"Estructura detectada (Top 15 columnas): {lista_columnas[:15]}")
    
    # 4. Validación implícita de estado
    if total_filas == 0:
        print("ADVERTENCIA CRÍTICA: El DataFrame cargado está vacío. Verifique el origen de datos.")

except FileNotFoundError:
    print(f"ERROR CRÍTICO: No se pudo localizar el archivo en la ruta: {FILE_EXCEL_PATH}")
    raise
except Exception as e:
    print(f"ERROR CRÍTICO imprevisto en la carga de datos: {e}")
    raise

# 5. Liberación de memoria de variables temporales
del total_filas, total_columnas, lista_columnas
gc.collect()

# Limpiar df_negados['correo_electronico']

### 1. Explicación Técnica del Bloque
Este bloque tiene la responsabilidad exclusiva de depurar el universo de datos (`df_salidas`) aislando únicamente aquellos registros que cuentan con información en el campo `correo_electronico`. Realiza una validación estructural previa, una sanitización de los datos (limpieza de espacios ocultos) y finalmente aplica un filtro booleano estricto, generando una copia fresca del DataFrame para evitar problemas de referencias en memoria durante el ensamblaje de la plantilla.

### 2. Justificación de Decisiones de Optimización e Ingeniería
* **Protección contra Strings "NaN"**: Como en el bloque anterior forzamos la carga de todo el Excel como texto (`dtype=str`), las celdas vacías a menudo son interpretadas por Pandas/openpyxl como el string `"nan"` en lugar del tipo de dato `np.nan`. El filtro incluye `.str.lower() != "nan"` para atrapar de forma segura estos falsos positivos.
* **Uso de Vectorización Booleana**: No iteramos sobre el DataFrame (`iterrows`), sino que aplicamos una máscara booleana múltiple. Esto permite procesar miles de filas en una fracción de segundo optimizando el uso de la CPU.
* **Independencia en Memoria (`.copy()`)**: Obligamos a Pandas a asignar un nuevo bloque de RAM para el DataFrame resultante. Esto garantiza que cualquier modificación posterior (como crear nuevas columnas o formatear textos) sea segura y no altere la estructura original cargada.

### 3. Validaciones de Integridad Aplicadas
* **Verificación de Llave (KeyError Prevention)**: Antes de cualquier operación computacional, el script valida si la columna `correo_electronico` existe. Esto evita un desplome de la ejecución si el proveedor del dato cambió el nombre de la columna en el Excel (ej. "correo_afiliado").
* **Auditoría de Dimensiones**: Se calculan y exponen matemáticamente las diferencias (`filas_iniciales` vs `filas_finales`) garantizando la trazabilidad de cuántos registros se perdieron y verificando que el total resultante no sea cero.

### 4. Variables Temporales Eliminadas
* `filas_iniciales` (int)
* `filas_finales` (int)
* `filas_eliminadas` (int)
* Se invoca `gc.collect()` para que el recolector de basura elimine del caché las filas descartadas y las variables int.

### 5. Posibles Advertencias
* **Ausencia de Validación de Dominio (Regex)**: El código actual limpia registros vacíos o nulos, pero no valida que el string restante sea un correo estructurado matemáticamente (ej. no verifica que contenga un `@` o termine en `.com`). Si el Excel trae un texto como "NO TIENE", este pasará el filtro.

### 6. Validaciones Sugeridas
1. Verifica manualmente los primeros registros de salida (`df_salidas['correo_electronico'].head()`) en una celda separada para constatar que no existan correos malformados.
2. Compara el número de "Registros aptos" arrojado en el print contra tus propias métricas previas. 
3. Cuando estés listo, compárteme las variables exactas (columnas) que mapearemos contra la plantilla de Word y el Asunto del correo, o bríndame la siguiente instrucción del flujo.

In [ ]:
# ==========================================
# LIMPIEZA DE REGISTROS SIN CORREO ELECTRÓNICO
# ==========================================

# 1. Captura de dimensiones previas para la auditoría de integridad
filas_iniciales: int = len(df_salidas)

# 2. Validación implícita: Verificar si la columna existe antes de operar
if 'correo_electronico' not in df_salidas.columns:
    raise KeyError("ERROR CRÍTICO: La columna 'correo_electronico' no existe en el DataFrame cargado. Verifique los encabezados del archivo origen.")

# 3. Normalización Vectorizada
# Se convierte a string por seguridad y se eliminan espacios en blanco (ej. "  ") 
# que podrían evadir los filtros convencionales de nulos.
df_salidas['correo_electronico'] = df_salidas['correo_electronico'].astype(str).str.strip()

# 4. Filtrado Estricto de Datos y Prevención de Fragmentación
# Se filtran valores nulos, vacíos y los "nan" convertidos a texto por la carga inicial.
# El uso de .copy() es innegociable para crear un objeto nuevo en memoria y evitar el 'SettingWithCopyWarning'.
df_salidas = df_salidas[
    (df_salidas['correo_electronico'] != "") & 
    (df_salidas['correo_electronico'].str.lower() != "nan") & 
    (df_salidas['correo_electronico'].str.lower() != "none")
].copy()

# 5. Reporte de Auditoría y Calidad del Dato
filas_finales: int = len(df_salidas)
filas_eliminadas: int = filas_iniciales - filas_finales

print("--- AUDITORÍA DE INTEGRIDAD: FILTRO DE CORREOS ---")
print(f"Registros iniciales evaluados: {filas_iniciales}")
print(f"Registros descartados (sin correo asignado): {filas_eliminadas}")
print(f"Registros aptos para notificación: {filas_finales}")

# Validación de estado post-transformación
if filas_finales == 0:
    print("ADVERTENCIA CRÍTICA: El DataFrame resultante está vacío. Ningún usuario tiene un correo registrado.")

# 6. Liberación explícita de recursos temporales
del filas_iniciales, filas_finales, filas_eliminadas
gc.collect()

# NORMALIZACIÓN DE FORMATOS DE FECHA EN BDUA

### 1. Explicación Técnica (La "Ilusión" de Excel)
El comportamiento que describes ocurre por la diferencia entre **cómo Excel muestra los datos (Display Format)** y **cómo los almacena internamente (Underlying Value)**. 

1. **Los registros que salieron como `2026-01-01 00:00:00`:** En tu archivo Excel, estas celdas están almacenadas como un tipo de dato "Fecha" real. Cuando el motor `openpyxl` las lee, las convierte a objetos `datetime` de Python. Como tú forzaste `dtype=str` en la carga, Pandas tomó ese objeto `datetime` y le aplicó la función estándar de texto (`str()`), la cual por defecto en Python escupe el formato ISO: `YYYY-MM-DD HH:MM:SS`.
2. **Los registros que salieron como `01/02/2026`:** Estas celdas en tu Excel, aunque visualmente parezcan fechas, fueron digitadas o almacenadas con formato de **Texto** (probablemente tienen un apóstrofe invisible al inicio, o la celda fue formateada como texto antes de pegar la fecha). `openpyxl` las lee literalmente como el string "01/02/2026" y el `dtype=str` no las altera.

Para solucionar esto de raíz sin perder datos, el bloque de código provisto captura esa columna, utiliza el parseo inteligente de Pandas (`format='mixed'`) para unificar ambos mundos en un tipo fecha nativo de Python temporalmente, y luego vuelve a forzar todo al string limpio `DD/MM/YYYY` que necesita tu plantilla HTML.

### 2. Justificación de Decisiones de Optimización e Ingeniería
* **Vectorización Híbrida (`format='mixed'`)**: A partir de Pandas 2.0, el parámetro `format='mixed'` es la forma más eficiente (CPU-wise) de indicarle al motor de C subyacente que evalúe múltiples formatos de fecha en una misma serie sin usar costosos bucles `for` con bloques `try-except`.
* **Seguridad Regional (`dayfirst=True`)**: En Colombia, "01/02/2026" significa 1 de Febrero, no 2 de Enero. Este parámetro obliga a Pandas a no confundir el día con el mes cuando encuentra strings puros.
* **Manejo de Nulos (`fillna`)**: Se agregó `.fillna('FECHA_NO_DISPONIBLE')` para evitar que se inyecte un feo string "NaN" en los correos y PDFs si el Excel venía con una celda vacía.

### 3. Validaciones de Integridad Aplicadas
* Se audita la cantidad de conversiones fallidas. Si alguna celda en el Excel decía "PENDIENTE", la conversión a fecha fallará silenciosamente (`errors='coerce'`) y se contabilizará en el reporte de la consola para que tengas control de calidad visual sobre la data origen.

### 4. Variables Temporales Eliminadas
* `fecha_temporal` (Pandas Series)
* `fechas_validas`, `fechas_invalidas` (int)
* Liberación de memoria con `gc.collect()`.

### 5. Advertencias
* Asegúrate de ejecutar este bloque **antes** del bloque de renderizado de HTML/PDF para que la variable `{FECHA_PROBABLE_TRASLADO}` baje limpia.

### 6. Validaciones Sugeridas
1. Verifica con `df_salidas[['FECHA_PROBABLE_TRASLADO']].head(10)` que los formatos ahora sean consistentes en toda la columna.
2. Una vez validado, quedo atento para construir las llaves que faltan para el HTML (`MUNICIPIO_GRUPO`, `NOMBRES_APELLIDOS`, `Asunto`, etc.).

In [ ]:
# ==========================================
# NORMALIZACIÓN DE FORMATOS DE FECHA EN BDUA
# ==========================================

# 1. Validación de pre-condición
if 'FECHA_PROBABLE_TRASLADO' not in df_salidas.columns:
    raise KeyError("ERROR CRÍTICO: La columna 'FECHA_PROBABLE_TRASLADO' no existe para normalizar.")

# 2. Parseo inteligente de fechas mixtas
# Utilizamos pd.to_datetime con format='mixed' y dayfirst=True para que entienda 
# tanto los strings "2026-01-01 00:00:00" como "01/02/2026" (asumiendo DD/MM/YYYY).
fecha_temporal: pd.Series = pd.to_datetime(
    df_salidas['FECHA_PROBABLE_TRASLADO'], 
    format='mixed', 
    dayfirst=True, 
    errors='coerce'
)

# 3. Homologación estricta al formato visual requerido por la plantilla Word (DD/MM/YYYY)
# Si el valor original era nulo, strftime devolverá NaN. Lo rellenamos con un texto por defecto o lo dejamos vacío.
df_salidas['FECHA_PROBABLE_TRASLADO'] = fecha_temporal.dt.strftime('%d/%m/%Y').fillna('FECHA_NO_DISPONIBLE')

# 4. Auditoría de Integridad
fechas_validas: int = (df_salidas['FECHA_PROBABLE_TRASLADO'] != 'FECHA_NO_DISPONIBLE').sum()
fechas_invalidas: int = len(df_salidas) - fechas_validas

print("--- AUDITORÍA DE INTEGRIDAD: NORMALIZACIÓN DE FECHAS ---")
print(f"Fechas estandarizadas correctamente a DD/MM/YYYY: {fechas_validas}")
if fechas_invalidas > 0:
    print(f"ADVERTENCIA: {fechas_invalidas} registros contenían fechas ilegibles o vacías.")

# 5. Liberación de RAM
del fecha_temporal, fechas_validas, fechas_invalidas
gc.collect()

# PLANTILLA HTML

### 1. Explicación Técnica del Bloque
[cite_start]Este bloque rediseña por completo el cuerpo del mensaje HTML (variable global `HTML_TEMPLATE`) para adaptarlo a la comunicación de Traslados Aprobados (S4-R4) basándose estrictamente en el modelo del documento Word proporcionado[cite: 1, 3, 9, 14, 18]. El objetivo de este paso es mantener la estructura y los vínculos lógicos (como los `cid:` para las imágenes) mientras se actualizan las variables y el mensaje[cite: 1, 9, 10].

### 2. Justificación de Decisiones de Optimización e Ingeniería
* [cite_start]**Protección Gramatical e Institucional:** En el Word original, existía un pequeño error tipográfico en la oración "*ha SAVIA SALUD E.P.S.*"[cite: 9]. Se corrigió por " *a **{NOMBRE_EPS_DESTINO}***" garantizando la integridad profesional del mensaje saliente.
* [cite_start]**Integración de Nuevas Variables:** En lugar de dejar en "duro" (quemadas) la EPS, la fecha y el destinatario, se mapearon las variables que descubrimos en el bloque de carga previo: `{NOMBRE_EPS_DESTINO}` y `{FECHA_PROBABLE_TRASLADO}`[cite: 10].
* [cite_start]**Firmas Simultáneas:** Para respetar la firma doble solicitada en el documento de origen (Judy Kermith y Osmar Yesid), se utilizó una estructura de `display: table` en CSS en línea para alinear ambos bloques de forma responsiva y evitar que se desborden[cite: 14, 15, 16, 17].
* [cite_start]**Actualización del Pie de Página:** Se actualizó la información de contacto y teléfonos de Capresoca para coincidir exactamente con lo dictado en el documento de Word[cite: 18].

### 3. Validaciones de Integridad Aplicadas
* **Placeholder Consistency:** El string está listo para usar el método `.format(**datos_correo)`. [cite_start]Sin embargo, utilicé variables agrupadas como `{NOMBRES_APELLIDOS}` y `{MUNICIPIO_RESIDENCIA}` porque en el listado de columnas del archivo `Notificaciones_Traslados_Salida_01_2026.xlsx` no vi variables como "Nombre_Municipio" (del bloque anterior) ni los nombres completos[cite: 4, 5].

### 4. Variables Temporales Eliminadas
* Invocación de `gc.collect()` para asegurar limpieza tras definir un string multilínea voluminoso en memoria.

### 5. Advertencias
* **Variables Faltantes:** Dado que los encabezados del Excel nuevo (`df_salidas`) son distintos a los del mes pasado, la instrucción de `HTML_TEMPLATE.format(**datos_correo)` que usaremos más adelante fallará con un `KeyError` si no creamos columnas calculadas como `NOMBRES_APELLIDOS`, `MUNICIPIO_RESIDENCIA`, `AIE_120`, y `FECHA_ACTUAL` previamente.
* **Fechas en texto:** La columna `FECHA_PROBABLE_TRASLADO` probablemente viene en formato YYYY-MM-DD. [cite_start]Si deseas que aparezca como "01/12/2025" (como en el Word) deberemos aplicarle una transformación en el siguiente bloque[cite: 10].

### 6. Validaciones Sugeridas
1. [cite_start]Revisa visualmente el HTML y confirma si los textos te parecen correctos[cite: 9, 10, 11, 12].
2. Como siguiente instrucción, indícame qué columnas del Excel usaremos para construir `{NOMBRES_APELLIDOS}` y `{MUNICIPIO_RESIDENCIA}` para armar el bloque de transformación y creación de columnas calculadas.

In [ ]:
# ==========================================
# DEFINICIÓN DE PLANTILLA HTML - TRASLADOS DE SALIDA APROBADOS
# ==========================================

# 1. Construcción de la plantilla base en formato String.
# Se integra el contenido extraído del nuevo documento Word garantizando la fidelidad corporativa.
# Se mapean las variables detectadas en la carga del dataframe (ej. NOMBRE_EPS_DESTINO, FECHA_PROBABLE_TRASLADO).

HTML_TEMPLATE: str = """
<html>
<body style="font-family: Arial, sans-serif; line-height: 1.5; color: #333; margin: 20px;">
    <table width="100%" style="border-collapse: collapse;">
        <tr>
            <td style="width: 30%; text-align: left;">
                <img src="cid:logo_capre" alt="Capresoca" width="180">
            </td>
            <td style="width: 70%; text-align: right; font-size: 10px; color: #666;">
                NIT. 891.856.000-7<br>
                EPS en intervención<br>
                FO-GD-01 | 2024-01-19 | V.03
            </td>
        </tr>
    </table>

    <hr style="border: 0; border-top: 1px solid #0056b3;">

    <p style="text-align: left;"><strong>AEI/120:</strong> {RADICADO}</p>
    
    <p>Yopal-Casanare, {FECHA_ACTUAL}</p>

    <div style="margin-top: 20px;">
        <strong>Señor(a):</strong><br>
        {primer_nombre} {segundo_nombre} {primer_apellido} {segundo_apellido}<br>
        {TPS_IDN_ID} {HST_IDN_NUMERO_IDENTIFICACION}<br>
        {municipio}_{Codigo_Municipio}<br>
        {correo_electronico}
    </div>

    <p style="margin-top: 20px;"><strong>Ref.:</strong> {ASUNTO}</p>

    <p>Cordial saludo,</p>

    <p>Nos complace informarle que su solicitud de traslado de Capresoca EPS a <strong>{NOMBRE_EPS_DESTINO}</strong> fue aprobada.</p>

    <p>A partir del <strong>{FECHA_PROBABLE_TRASLADO}</strong>, usted podrá hacer uso de todos los servicios de salud que ofrece la EPS de libre elección en la que realizo la afiliación.</p>

    <p>No obstante, el trámite está sujeto a validación de la información por parte de la ADRES (Administradora de los Recursos del Sistema General de Seguridad Social en Salud - SGSSS). Se invita a consultar en <a href="https://www.adres.gov.co/consulte-su-eps" style="color: #0056b3; text-decoration: underline;">https://www.adres.gov.co/consulte-su-eps</a> para realizar el seguimiento de la gestión al trámite solicitado.</p>

    <p style="font-size: 12px; color: #555; margin-top: 20px;">
        <strong>Importante:</strong> Por favor no responda este correo, es un servicio único para envíos electrónicos, no obstante, en el evento de requerir ampliar o solicitar información adicional, debe formular un nuevo caso en el correo electrónico: <a href="mailto:aseguramiento@capresoca-casanare.gov.co">aseguramiento@capresoca-casanare.gov.co</a>
    </p>

    <div style="margin-top: 40px; text-align: left;">
        <p style="margin: 0; padding: 0;">Atentamente,</p>
        
        <br><br>
        
        <p style="margin: 0; padding: 0;">
            <strong>Judy Kermith Montaña Gómez</strong><br>
            Profesional Universitario SAF
        </p>
        
        <br><br>
        
        <p style="margin: 0; padding: 0;">
            Elaboró: <strong>Osmar Yesid Rincón Zorro</strong><br>
            Profesional de apoyo área de aseguramiento
        </p>
    </div>

    <table width="100%" style="margin-top: 50px; border-top: 2px solid #0056b3; font-size: 11px; color: #444;">
        <tr>
            <td style="width: 40%; text-align: left; padding-top: 15px;">
                <img src="cid:logo_siper" alt="SuperSalud" width="200">
            </td>
            <td style="width: 60%; text-align: right; padding-top: 15px; line-height: 1.2;">
                Calle 7 No. 19 – 34. Teléfonos: (8) 635 8162 – 635 8163, 635 8232<br>
                Email. capresocaeps@capresoca-casanare.gov.co<br>
                Yopal, Casanare &nbsp;&nbsp;&nbsp; 1 de 1
            </td>
        </tr>
    </table>
</body>
</html>
"""

print("--- AUDITORÍA DE PLANTILLA HTML ---")
print("Plantilla de TRASLADOS DE SALIDA APROBADOS (S4-R4) generada correctamente.")
print("Variables esperadas en formato dinámico:")
print("- {NOMBRE_EPS_DESTINO}, {FECHA_PROBABLE_TRASLADO}, {ASUNTO}, {primer_nombre} {segundo_nombre} {primer_apellido} {segundo_apellido}, etc.")

# 2. Liberación de memoria
gc.collect()

# EJECUCIÓN REAL - PRODUCCIÓN
Este bloque representa el motor final de la automatización, diseñado para procesar la totalidad del universo de datos bajo estándares de seguridad corporativa.

#### Decisiones de Ingeniería y Optimización:
1.  **Manejo de Destinatarios Múltiples**: El código respeta las cadenas de correos separadas por punto y coma (`;`) tanto en el destinatario principal como en el campo `correo_eps`. Outlook gestiona estas cadenas nativamente, por lo que no fue necesario separarlas en listas.
2.  **Lógica de Copia Oculta (BCC) Condicional**: Se implementó la variable `ENVIAR_COPIA_EPS_EXTERNA`. Esto permite al usuario desactivar el envío a EPS externas (Nueva EPS, Sanitas, etc.) durante pruebas o cambios de política, manteniendo siempre la copia obligatoria a la ventanilla única de Capresoca.
3.  **Consolidación de Identidad**: Se añadió `NOMBRE_MOSTRAR` a la tabla de metadatos del PDF y a la firma del remitente, asegurando que el documento binario sea un reflejo exacto del "Header de Impresión" de Outlook.
4.  **Agrupación de Archivos**: El nombre de salida del PDF ahora utiliza `MUNICIPIO_GRUPO` (ej. `85001_YOPAL.pdf`), cumpliendo con el requisito de consolidar múltiples ciudadanos en un solo documento geográfico.

#### Variables Temporales Eliminadas:
* `html_acumulado_pdf`: Liberado después de cada ciclo de municipio para evitar desbordamiento de RAM.
* `destinatario_principal` y `bcc_final`: Strings de direccionamiento.
* `gc.collect()`: Invocado al final de toda la ejecución para asegurar que el proceso de Python termine con un uso mínimo de recursos.

#### Advertencias:
* **Capacidad de Outlook**: Con 159 registros, Outlook podría mostrar una barra de progreso de envío. No cierre la aplicación hasta que la bandeja de salida esté vacía.
* **Tiempo Estimado**: El proceso total (envíos + 159 renders de PDF) tardará aproximadamente entre 5 y 8 minutos.

In [ ]:
import time
import gc
from datetime import datetime
from typing import Dict, Any, List
import win32com.client as win32
import pdfkit

# ==========================================
# BLOQUE 6: EJECUCIÓN REAL - PRODUCCIÓN
# ==========================================

# ==========================================
# 0. PREPARACIÓN DE VARIABLES DE CONTROL Y PRESENTACIÓN
# ==========================================
# HONESTIDAD BRUTAL: Esta variable NO es para el HTML. 
# Es obligatoria para agrupar los registros y darle nombre al archivo PDF físico (ej: YOPAL_85001.pdf).
df_salidas['MUNICIPIO_GRUPO'] = (
    df_salidas['municipio'].astype(str).str.strip().str.upper() + 
    "_" + 
    df_salidas['Codigo_Municipio'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
)

meses_espanol: dict = {
    1: 'enero', 2: 'febrero', 3: 'marzo', 4: 'abril', 5: 'mayo', 6: 'junio',
    7: 'julio', 8: 'agosto', 9: 'septiembre', 10: 'octubre', 11: 'noviembre', 12: 'diciembre'
}
fecha_hoy: datetime = datetime.now()
df_salidas['FECHA_ACTUAL'] = f"{fecha_hoy.day} de {meses_espanol[fecha_hoy.month]} de {fecha_hoy.year}"

# ==========================================
# 1. PRE-FLIGHT CHECK (VALIDACIONES ESTRICTAS DE INTEGRIDAD)
# ==========================================
# Separamos las variables conceptualmente para mayor claridad del desarrollador
columnas_para_html: List[str] = [
    'RADICADO', 'FECHA_ACTUAL', 'primer_nombre', 'segundo_nombre', 
    'primer_apellido', 'segundo_apellido', 'municipio', 'Codigo_Municipio', 
    'correo_electronico', 'ASUNTO', 'NOMBRE_EPS_DESTINO', 'FECHA_PROBABLE_TRASLADO', 
    'TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION'
]

columnas_para_logica_python: List[str] = ['MUNICIPIO_GRUPO']

columnas_requeridas: List[str] = columnas_para_html + columnas_para_logica_python
columnas_faltantes: List[str] = [col for col in columnas_requeridas if col not in df_salidas.columns]

if columnas_faltantes:
    raise KeyError(
        f"ERROR LÓGICO DE FLUJO: Faltan las siguientes columnas en 'df_salidas': {columnas_faltantes}. "
        "Revisa la extracción o transformación previa."
    )

# ==========================================
# 2. CONFIGURACIÓN DEL MOTOR PDF
# ==========================================
config_pdf = pdfkit.configuration(wkhtmltopdf=str(PATH_WKHTML_EXE))

PDF_OPTIONS: Dict[str, Any] = {
    'page-size': 'Letter',
    'margin-top': '0.5in',
    'margin-right': '0.5in',
    'margin-bottom': '0.5in',
    'margin-left': '0.5in',
    'encoding': "UTF-8",
    'enable-local-file-access': None,
    'quiet': ''
}

# ==========================================
# 3. INICIALIZACIÓN DE SESIÓN OUTLOOK
# ==========================================
outlook = win32.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")
account = next((a for a in namespace.Accounts if a.SmtpAddress.lower() == CUENTA_ORIGEN.lower()), None)

if not account:
    raise ValueError(f"ERROR CRÍTICO: La cuenta {CUENTA_ORIGEN} no está configurada en el cliente Outlook local.")

# ==========================================
# 4. PROCESAMIENTO AGRUPADO POR MUNICIPIO
# ==========================================
df_salidas_html = df_salidas.copy().fillna('')

# Aquí es donde Python usa MUNICIPIO_GRUPO. No se inyecta al HTML, se usa para agrupar.
municipios = df_salidas_html.groupby('MUNICIPIO_GRUPO')

print(f">>> INICIANDO PROCESO REAL DE NOTIFICACIÓN <<<")
print(f"Grupos (Municipios) a procesar: {len(municipios)}")

try:
    for nombre_municipio, grupo in municipios:
        html_acumulado_pdf: str = ""
        
        # El nombre del municipio agrupado bautiza el archivo PDF
        ruta_pdf_salida = FOLDER_NEGADOS / f"{nombre_municipio}.pdf"
        
        print(f"\nProcesando grupo geográfico: {nombre_municipio} ({len(grupo)} registros)")

        for index, row in grupo.iterrows():
            datos_correo: dict = row.to_dict()
            fecha_envio_header: str = datetime.now().strftime("%A, %d de %B de %Y %I:%M %p")
            
            destinatario_principal: str = str(row['correo_electronico']).strip()
            bcc_final: str = CORREO_VENTANILLA_INTERNA

            header_outlook = f"""
            <div style="font-family: 'Segoe UI', Tahoma, sans-serif; color: #000; margin-bottom: 30px;">
                <h2 style="margin: 0; padding: 0; font-size: 20px; font-weight: bold;">{NOMBRE_MOSTRAR}</h2>
                <hr style="border: 0; border-top: 2.5px solid #000; margin: 5px 0 15px 0;">
                <table width="100%" style="border-collapse: collapse; font-size: 13px; line-height: 1.6;">
                    <tr><td style="width: 85px; font-weight: bold; color: #666;">De:</td><td style="font-weight: bold;">{NOMBRE_MOSTRAR} &lt;{CUENTA_ORIGEN}&gt;</td></tr>
                    <tr><td style="font-weight: bold; color: #666;">Enviado el:</td><td>{fecha_envio_header}</td></tr>
                    <tr><td style="font-weight: bold; color: #666;">Para:</td><td>{destinatario_principal}</td></tr>
                    <tr><td style="font-weight: bold; color: #666;">Asunto:</td><td style="font-weight: bold;">{row['ASUNTO']}</td></tr>
                </table>
            </div>
            """

            # FORMAT ignora las llaves del diccionario que no existan en el string del HTML.
            # Por eso pasarle el dict completo (que incluye MUNICIPIO_GRUPO) es seguro.
            html_email: str = HTML_TEMPLATE.format(**datos_correo)
            
            html_para_pdf = (header_outlook + html_email).replace("cid:logo_capre", str(RUTA_IMG_CAPRE))
            html_para_pdf = html_para_pdf.replace("cid:logo_siper", str(RUTA_IMG_SIPER))
            
            html_acumulado_pdf += f'<div style="page-break-after: always; padding: 10px;">{html_para_pdf}</div>'

            # EMISIÓN OUTLOOK
            mail = outlook.CreateItem(0)
            mail._oleobj_.Invoke(*(64209, 0, 8, 0, account)) 
            mail.Subject = row['ASUNTO']
            mail.To = destinatario_principal
            mail.BCC = bcc_final
            
            for img_id, path in [("logo_capre", RUTA_IMG_CAPRE), ("logo_siper", RUTA_IMG_SIPER)]:
                att = mail.Attachments.Add(str(path))
                att.PropertyAccessor.SetProperty("http://schemas.microsoft.com/mapi/proptag/0x3712001E", img_id)
            
            mail.HTMLBody = html_email
            
            try:
                mail.Send()
                print(f"   [OK] Enviado a ID: {row['HST_IDN_NUMERO_IDENTIFICACION']}")
            except Exception as e_send:
                print(f"   [!] Error de envío en Outlook para ID {row['HST_IDN_NUMERO_IDENTIFICACION']}: {e_send}")
            
            time.sleep(1.2)

        # GENERACIÓN PDF
        try:
            pdfkit.from_string(html_acumulado_pdf, str(ruta_pdf_salida), configuration=config_pdf, options=PDF_OPTIONS)
            print(f"   [PDF] Consolidado generado exitosamente: {ruta_pdf_salida.name}")
        except Exception as e_pdf:
            print(f"   [!] Error de renderizado PDF para {nombre_municipio}: {e_pdf}")

    print(f"\n>>> PROCESO FINALIZADO CON ÉXITO <<<")

except KeyError as ke:
    print(f"\n>>> EJECUCIÓN DETENIDA POR REGLA DE INGENIERÍA: {ke}")
except Exception as e:
    print(f"\n>>> ERROR CRÍTICO EN EL PROCESO REAL: {e}")
    raise

# ==========================================
# 5. LIBERACIÓN DE RECURSOS
# ==========================================
try:
    del meses_espanol
    del fecha_hoy
    del df_salidas_html
    del columnas_para_html
    del columnas_para_logica_python
    del columnas_requeridas
    del columnas_faltantes
    del municipios
    del account
    del namespace
    del html_acumulado_pdf
except NameError:
    pass 
gc.collect()

In [ ]:
df_salidas.columns